In [28]:
# importing libraries
from pathlib import Path
import json

import numpy as np
import pandas as pd
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

print("All preprocessing libraries imported successfully.")

All preprocessing libraries imported successfully.


In [29]:
# loading the saved datasets

PROJECT_ROOT = Path.cwd().resolve().parent

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_data.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test_data.csv"

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

TARGET = "Class"

FEATURES = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRation", "Eccentricity", "ConvexArea", "EquivDiameter",
    "Extent", "Solidity", "roundness", "Compactness",
    "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"
]

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training classes:", y_train.nunique())

assert X_train.shape[1] == 16
assert X_test.shape[1] == 16
assert y_train.nunique() == 7

Training features: (10834, 16)
Testing features: (2709, 16)
Training classes: 7


In [30]:
# importing our custom functions 
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import build_preprocessor, FEATURES

print("Reusable preprocessing module imported.")

ImportError: cannot import name 'build_preprocessor' from 'src.preprocessing' (/Users/mohdaakilkhan/Documents/Aakil_MAC/BITS_Mtech/BITS MTECH Learning/SEM-1/ML/2026AJ05523_ML_Assignment_2/src/preprocessing.py)

In [ ]:
# build preprocessing pipeline for all five models
MODEL_CONFIGS = {
    "Logistic Regression": {
        "estimator": LogisticRegression(
            max_iter=2000,
            random_state=42
        ),
        "scale": True
    },

    "Decision Tree": {
        "estimator": DecisionTreeClassifier(
            random_state=42
        ),
        "scale": False
    },

    "KNN": {
        "estimator": KNeighborsClassifier(
            n_neighbors=5
        ),
        "scale": True
    },

    "Naive Bayes": {
        "estimator": GaussianNB(),
        "scale": False
    },

    "SVM": {
        "estimator": LinearSVC(
            max_iter=5000,
            random_state=42
        ),
        "scale": True
    }
}

MODEL_PIPELINES = {}

for model_name, config in MODEL_CONFIGS.items():

    pipeline = Pipeline([
        (
            "preprocessing",
            build_preprocessor(scale=config["scale"])
        ),
        (
            "classifier",
            clone(config["estimator"])
        )
    ])

    MODEL_PIPELINES[model_name] = pipeline

print("Pipelines created:", len(MODEL_PIPELINES))

for name, pipeline in MODEL_PIPELINES.items():
    print(f"\n{name}")
    print(pipeline)

In [ ]:
# fit preprocessing data on traning data only 
scaled_preprocessor = build_preprocessor(scale=True)

scaled_preprocessor.fit(X_train)

X_train_scaled = scaled_preprocessor.transform(X_train)

print("Scaled training shape:", X_train_scaled.shape)

print("First 5 transformed rows:")
print(X_train_scaled[:5])

In [ ]:
# verification of sacling
scaled_means = X_train_scaled.mean(axis=0)
scaled_stds = X_train_scaled.std(axis=0)

print("Means after scaling:")
print(np.round(scaled_means, 4))

print("\nStandard deviations after scaling:")
print(np.round(scaled_stds, 4))

In [ ]:
# Inspection of the fitted parameter
numeric_pipeline = (
    scaled_preprocessor
    .named_transformers_["numeric"]
)

imputer = numeric_pipeline.named_steps["imputer"]
scaler = numeric_pipeline.named_steps["scaler"]

parameters = pd.DataFrame({
    "Feature": FEATURES,
    "Median": imputer.statistics_,
    "Mean": scaler.mean_,
    "Scale": scaler.scale_
})

display(parameters)

In [ ]:
# saving preprocessing artifact
unscaled_preprocessor = build_preprocessor(scale=False)

unscaled_preprocessor.fit(X_train)

scaled_path = MODEL_DIR / "preprocessor_scaled.joblib"
unscaled_path = MODEL_DIR / "preprocessor_unscaled.joblib"

joblib.dump(
    scaled_preprocessor,
    scaled_path
)

joblib.dump(
    unscaled_preprocessor,
    unscaled_path
)

print("Saved:", scaled_path)
print("Saved:", unscaled_path)

In [ ]:
# reload and verify
loaded_scaled = joblib.load(scaled_path)

original_output = scaled_preprocessor.transform(
    X_train.head(10)
)

loaded_output = loaded_scaled.transform(
    X_train.head(10)
)

np.testing.assert_allclose(
    original_output,
    loaded_output
)

print("Saved and reloaded preprocessing outputs match.")

In [ ]:
# testing preprocessing without using test labels
X_test_scaled = loaded_scaled.transform(X_test)

print("Test feature shape:", X_test.shape)
print("Transformed test shape:", X_test_scaled.shape)

assert X_test_scaled.shape[1] == 16
assert np.isfinite(X_test_scaled).all()

print("Test feature transformation successful.")

In [ ]:
# preapring for cross validtion 

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold_number, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train),
    start=1
):
    print(
        f"Fold {fold_number}: "
        f"Train={len(train_idx)}, "
        f"Validation={len(val_idx)}"
    )

## Preprocessing Decisions

### Feature Schema
The dataset contains 16 numerical predictors and one categorical target (`Class`). All five classifiers use the same feature set and stratified train/test split.

### Missing Values
Median imputation is included in the pipelines for robustness. Imputation parameters are fitted using training observations only.

### Scaling
StandardScaler is used for Logistic Regression, KNN and SVM because these models are sensitive to feature magnitudes.

Decision Tree and Gaussian Naive Bayes use median imputation without scaling.

### Data Leakage Prevention
Preprocessing and classification are combined into scikit-learn pipelines. During cross-validation, preprocessing will be fitted separately on each training fold.

The held-out test set will not be used for hyperparameter selection.

### Reproducibility
Random states are fixed where applicable. Standalone preprocessing artifacts are saved and verified through serialization tests.

### Next Step
Train all five classifiers, measure training and prediction times, and compare validation performance before final held-out test evaluation.